In [2]:
pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [17]:
import os
from dotenv import load_dotenv

un = os.getenv("user")
pw = os.getenv("password")
cs = os.getenv("connstr")
load_dotenv()

# Connect to the database
import oracledb

connection = oracledb.connect(user=un, password=pw, dsn=cs)

table_name = 'faqs'

In [18]:
import os

def loadFAQs(directory_path):
    faqs = {}

    for filename in os.listdir(directory_path):
        #print("directory_path: " + directory_path)
        #print("filename: " + filename)
        #print("\n")
        
        if filename.endswith("faq.txt"):    # assuming FAQs are in .txt files
            file_path = os.path.join(directory_path, filename)

            with open(file_path) as f:
                raw_faq = f.read()    # Only read ANSI format file

            filename_without_ext = os.path.splitext(filename)[0]    #remove .txt extention
            faqs[filename_without_ext] = [text.strip() for text in raw_faq.split('=====')]

            print("directory_path: " + directory_path)
            print("filename: " + filename)
            print("\n")

    return faqs

In [19]:
faqs = loadFAQs('D:/LLM')
faqs

directory_path: D:/LLM
filename: faq.txt




{'faq': ['What is Oracle Cloud Free Tier?  \n \nOracle Cloud Free Tier allows you to sign up for an Oracle Cloud \naccount which provides a number of Always Free services and a \nFree Trial with US$300 of free credit to use on all eligible \nOracle Cloud Infrastructure services for up to 30 days. The \nAlways Free services are available for an unlimited period of \ntime. The Free Trial services may be used until your US$300 of \nfree credits are consumed or the 30 days has expired, whichever \ncomes first.',
  'Who should use Oracle Cloud Free Tier?  \n \nOracle Cloud Free Tier services are for everyone. Whether you’re \na developer building and testing applications, a startup founder \ncreating new systems with the intention of scaling later, an \nenterprise looking to test things before moving to cloud, a \nstudent wanting to learn, or an academic developing curriculum \nin the cloud, Oracle Cloud Free Tier enables you to learn, \nexplore, build and test for free.',
  'Why do I need 

In [20]:
docs = [{'text': filename + ' | ' + section, 'path': filename} for filename, sections in faqs.items() for section in sections]

# Sample the resulting data
docs[:2]


[{'text': 'faq | What is Oracle Cloud Free Tier?  \n \nOracle Cloud Free Tier allows you to sign up for an Oracle Cloud \naccount which provides a number of Always Free services and a \nFree Trial with US$300 of free credit to use on all eligible \nOracle Cloud Infrastructure services for up to 30 days. The \nAlways Free services are available for an unlimited period of \ntime. The Free Trial services may be used until your US$300 of \nfree credits are consumed or the 30 days has expired, whichever \ncomes first.',
  'path': 'faq'},
 {'text': 'faq | Who should use Oracle Cloud Free Tier?  \n \nOracle Cloud Free Tier services are for everyone. Whether you’re \na developer building and testing applications, a startup founder \ncreating new systems with the intention of scaling later, an \nenterprise looking to test things before moving to cloud, a \nstudent wanting to learn, or an academic developing curriculum \nin the cloud, Oracle Cloud Free Tier enables you to learn, \nexplore, build

In [21]:
# Connection detailes
#un = os.getenv("user", "TESTDB")
#pw = os.getenv("password", "?renoma1")
#cs = os.getenv("connstr", "localhost:1521/FREEPDB1")

# Connect to the database
#import oracledb

#connection = oracledb.connect(user=un, password=pw, dsn=cs)

#table_name = 'faqs'

with connection.cursor() as cursor:
    # Create the table
    create_table_sql = f"""
        CREATE TABLE IF NOT EXISTS {table_name} (
            id NUMBER PRIMARY KEY,
            payload CLOB CHECK (payload IS JSON),
            vector VECTOR
            )"""
    try:
        cursor.execute(create_table_sql)
    except oracledb.DatabaseError as e:
        raise

    connection.autocommit = True

In [22]:
from sentence_transformers import SentenceTransformer
encoder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', local_files_only=True)

import array

# Define a list to store the data
data = [
    {"id": idx, "vector_source": row['text'], "payload": row}
    for idx, row in enumerate(docs)
]

# Collect all text for batch encoding
texts = [f"{row['vector_source']}" for row in data]

# Encode all texts in a batch
embeddings = encoder.encode(texts, batch_size=32, show_progress_bar=True)

# Assign the embedding back to your data structure
for row, embedding in zip(data, embeddings):
    row['vector'] = array.array("f", embedding)    

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [23]:
import json

with connection.cursor() as cursor:
    # Truncate the table
    cursor.execute(f"TRUNCATE TABLE {table_name}")

    prepared_data = [(row['id'], json.dumps(row['payload']), row['vector']) for row in data]

    # Insert the data
    cursor.executemany(
        f"""INSERT INTO {table_name} (id, payload, vector)
        VALUES (:1, :2, :3)""",
        prepared_data
    )

    connection.commit()

In [ ]:
with connection.cursor() as cursor:
    # Define the query to select all rows from a table
    query = f"""SELECT *
                  FROM {table_name}"""

    # Execute the query
    cursor.execute(query)

    # Fetch all rows
    rows = cursor.fetchall()

    # Print the rows
    for row in rows[:5]:
        print(row)

In [61]:
# Define the SQL script used to retrive the chunk
topK = 2

sql = f"""SELECT payload, 
                 VECTOR_DISTANCE(vector, :vector, COSINE) as score
            FROM {table_name}
           ORDER BY score
           FETCH APPROX FIRST {topK} ROWS ONLY"""

# Transform the question into a vector
question = "What is a Always Free? How long using period?"

# Execute the query
with connection.cursor() as cursor:
    embedding = list(encoder.encode(question))
    vector = array.array("f", embedding)

    results = []

    for (info, score) in cursor.execute(sql, vector=vector):
        text_content = info.read()
        results.append((score, json.loads(text_content)))

# Print the reslts
import pprint
pprint.pp(results)

[(0.6864040123259973,
  {'text': 'faq | What is Oracle Cloud Free Tier?  \n'
           ' \n'
           'Oracle Cloud Free Tier allows you to sign up for an Oracle Cloud \n'
           'account which provides a number of Always Free services and a \n'
           'Free Trial with US$300 of free credit to use on all eligible \n'
           'Oracle Cloud Infrastructure services for up to 30 days. The \n'
           'Always Free services are available for an unlimited period of \n'
           'time. The Free Trial services may be used until your US$300 of \n'
           'free credits are consumed or the 30 days has expired, whichever \n'
           'comes first.',
   'path': 'faq'}),
 (0.7460706758094497,
  {'text': 'faq | Why do I need to provide credit or debit card information '
           'when I \n'
           'sign up for Oracle Cloud Free Tier?  \n'
           ' \n'
           'To provide free Oracle Cloud accounts to our valued customers, \n'
           'we need to ensure that you

In [ ]:
pip install -U langchain-ollama

In [66]:
#from langchain_ollama import OllamaLLM
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

LLM_MODEL = "gemma3:1b"

# Model
llm = ChatOllama(
        model=LLM_MODEL,
        temperature=0.1,
        base_url="http://localhost:11434",
        top_k=1
    )

# 프롬프트 템플릿 설정 (선택 사항이지만 권장)
prompt = ChatPromptTemplate.from_messages([
    ("system", "This is first my RAG."),
    ("user", "{input}")
])

# 출력 파서 설정 (LLM의 출력을 문자열로 파싱)
output_parser = StrOutputParser()

# 체인 구성
chain = prompt | llm | output_parser

# 답변 요청

# 검색된 문서 리스트를 문자열로 변환하여 LLM 프롬프트에 포함
prompt_with_rag = f"""
    RAG Content:\n
    {json.dumps(results, ensure_ascii=False, indent=2)} \n\n
    Question:\n {question}
"""

#question = "대한민국의 수도는 어디인가요?"
response = chain.invoke({"input": prompt_with_rag})

# 답변 출력
print(f"Question: {question}")
print(f"Answer: {response}")

Question: What is a Always Free? How long using period?
Answer: According to the RAG content:

*   **Always Free:** Oracle Cloud Free Tier provides a number of Always Free services and a Free Trial with US$300 of free credit to use on all eligible Oracle Cloud Infrastructure services for up to 30 days.
*   **Time Period:** The Free Trial services may be used until your US$300 of free credits are consumed or the 30 days has expired, whichever comes first.


In [30]:
from transformers import LlamaTokenizerFast
import sys

tokenizer = LlamaTokenizerFast.from_pretrained("hf-internal-testing/llama-tokenizer", local_files_only=True)

tokenizer.model_max_length = sys.maxsize

def truncate_string(string, max_tokens):
    # Tokenize the text and count th token
    tokens = tokenizer.encode(string, add_special_tokens=True)
    # Truncate the tokens to a maximum length
    truncated_tokens = tokens[:max_tokens]
    # Transform the tokens baxk to text
    truncated_text = tokenizer.decode(truncated_tokens)
    
    return truncated_text

In [44]:
# Transform doc into a string array using the "payload" key

print(docs)
print("---------------------------------------------")

#doc_as_one_string = "\n=========\n".join([doc["text"] for doc in docs])
doc_as_one_string = "\n**********\n".join([doc["text"] for doc in docs])  #Document당 구분가능한 token 삽입
docs_truncated = truncate_string(doc_as_one_string, 1000)  #Embedding Model에서 처리 가능한 token수로 절단

print(docs_truncated)

[{'text': 'faq | What is Oracle Cloud Free Tier?  \n \nOracle Cloud Free Tier allows you to sign up for an Oracle Cloud \naccount which provides a number of Always Free services and a \nFree Trial with US$300 of free credit to use on all eligible \nOracle Cloud Infrastructure services for up to 30 days. The \nAlways Free services are available for an unlimited period of \ntime. The Free Trial services may be used until your US$300 of \nfree credits are consumed or the 30 days has expired, whichever \ncomes first.', 'path': 'faq'}, {'text': 'faq | Who should use Oracle Cloud Free Tier?  \n \nOracle Cloud Free Tier services are for everyone. Whether you’re \na developer building and testing applications, a startup founder \ncreating new systems with the intention of scaling later, an \nenterprise looking to test things before moving to cloud, a \nstudent wanting to learn, or an academic developing curriculum \nin the cloud, Oracle Cloud Free Tier enables you to learn, \nexplore, build an

In [32]:
prompt = f"""
    <s>[INST] <<SYS>> 
    You are a helpful assistant named Oracle chatbot.  
    USE ONLY the sources below and ABSOLUTELY IGNORE any previous 
knowledge. 
    Use Markdown if appropriate. 
    Assume the customer is highly technical. 
    <</SYS>> [/INST] 
 
    [INST] 
    Respond to PRECISELY to this question: "{question}.",  USING ONLY 
the following information and IGNORING ANY PREVIOUS KNOWLEDGE. 
    Include code snippets and commands where necessary. 
    NEVER mention the sources, always respond as if you have that 
knowledge yourself. Do NOT provide warnings or disclaimers. 
    ===== 
    Sources: {docs_truncated} 
    ===== 
    Answer (Three paragraphs, maximum 50 words each, 90% spartan): 
    [/INST] 
    """